In [2]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


### CÀI ĐẶT HÀM

In [76]:
import numpy as np

def conjugate_gradient(A, b, x0, eps) -> tuple[np.ndarray, int]:
    """
    A: Ma trận vuông, đối xứng xác định dương.
    b: Vectơ vế phải.
    x0: Vectơ nghiệm ban đầu.
    eps: Sai số cho phép, eps > 0.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.asarray(x0, dtype=float).copy()

    # Kiểm tra ma trận vuông
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A phải là ma trận vuông không rỗng.")

    n = A.shape[0]

    # Kiểm tra ma trận đối xứng xác định dương
    if not np.allclose(A, A.T):
        raise ValueError("A phải là ma trận đối xứng.")

    for i in range(1, n + 1):
        det = np.linalg.det(A[:i, :i])
        if det <= 0:
           raise ValueError("A không phải ma trận xác định dương.")
        
    # Khởi tạo phần dư và hướng tìm kiếm
    r = b - np.dot(A, x)
    s = r.copy()

    print("Nghiệm ban đầu x0:", x)
    print("Phần dư ban đầu r0:", r)

    if np.max(np.abs(r)) < eps:
        print("Nghiệm ban đầu đã thỏa mãn điều kiện dừng.")
        return x, 0

    for k in range(1, n + 1):

        # Tính alpha
        As = np.dot(A, s)
        mau = np.dot(s, As)

        if mau <= 0 or not np.isfinite(mau):
            raise ValueError("Mẫu số không hợp lệ.")

        alpha = np.dot(s, r) / mau

        # Cập nhật nghiệm và phần dư
        x = x + alpha * s
        r_moi = b - np.dot(A, x)
        chuan_r = np.sqrt(np.sum(r_moi**2))

        # Hiển thị kết quả từng lần lặp
        print(f"\nLần lặp thứ {k}:")
        print("Nghiệm x =", x)
        print("Phần dư r =", r_moi)
        print("Chuẩn phần dư ||r|| =", chuan_r)

        # Kiểm tra điều kiện dừng
        if chuan_r < eps:
            print(f"Thỏa mãn sau {k} lần lặp.")
            return x, k

        # Tính beta
        beta = -np.dot(r_moi, np.dot(A, s)) / mau

        # Cập nhật hướng tìm kiếm
        s = r_moi + beta * s
        r = r_moi.copy()

    print(f"Không hội tụ sau {n} lần lặp.")
    return x, n


In [69]:
# Ví dụ 4.10
A = np.array([[4, -1,  1],
              [-1, 4, -2],
              [1, -2,  4]], dtype=float)
b = np.array([12, -1, 5], dtype=float)
kq = conjugate_gradient(A, b, x0 = np.array([0, 0, 0]) , eps = 1e-4)

Nghiệm ban đầu x0: [0. 0. 0.]
Phần dư ban đầu r0: [12. -1.  5.]

Lần lặp thứ 1:
Nghiệm x = [ 2.41706161 -0.2014218   1.007109  ]
Phần dư r = [ 1.12322275  4.23696682 -1.84834123]
Chuẩn phần dư ||r|| = 4.757087609803571

Lần lặp thứ 2:
Nghiệm x = [3.07752283 0.79483181 0.71998218]
Phần dư r = [-0.2352417   0.33815995  0.63221207]
Chuẩn phần dư ||r|| = 0.7545746578244767

Lần lặp thứ 3:
Nghiệm x = [3. 1. 1.]
Phần dư r = [-1.77635684e-15  8.88178420e-16 -1.77635684e-15]
Chuẩn phần dư ||r|| = 2.6645352591003757e-15
Thỏa mãn sau 3 lần lặp.


### Kiểm chứng: gradient hội tụ không quá n bước lặp

In [78]:
def kiem_chung(A, b, x0, eps=1e-8):

    # Lấy cấp ma trận
    n = A.shape[0]

    # Gọi thuật toán Gradient liên hợp
    x, k = conjugate_gradient(A, b, x0, eps)
    
    # Hiển thị kết quả
    print("\n= KIỂM CHỨNG HỘI TỤ =")
    print("Cấp ma trận n:", n)
    print("Số lần lặp k:", k)

    # Kiểm tra điều kiện
    if k <= n:
        print("KẾT LUẬN: Thuật toán hội tụ không quá n bước.")
    else:
        print("KẾT LUẬN: Chưa kiểm chứng được tính hội tụ.")

    return x, k
    
# Xét các ví dụ
# Ma trận cấp 3

A3 = np.array([
    [4, -1,  1],
    [-1, 4, -2],
    [1, -2,  4]
], dtype=float)

b3 = np.array([12, -1, 5], dtype=float)
x03 = np.zeros(3)

print("\nVÍ DỤ CẤP 3")
kiem_chung(A3, b3, x03)

# Ma trận cấp 4

A4 = np.array([
    [4, -1,  0,  0],
    [-1, 4, -1,  0],
    [0, -1,  4, -1],
    [0,  0, -1,  4]
], dtype=float)

b4 = np.array([2, 4, 6, 13], dtype=float)
x04 = np.zeros(4)

print("\nVÍ DỤ CẤP 4")
kiem_chung(A4, b4, x04)

# Ma trận cấp 5

A5 = np.array([
    [4, -1,  0,  0,  0],
    [-1, 4, -1,  0,  0],
    [0, -1,  4, -1,  0],
    [0,  0, -1,  4, -1],
    [0,  0,  0, -1,  4]
], dtype=float)

b5 = np.array([2, 4, 6, 8, 16], dtype=float)
x05 = np.zeros(5)

print("\nVÍ DỤ CẤP 5")
kiem_chung(A5, b5, x05)



VÍ DỤ CẤP 3
Nghiệm ban đầu x0: [0. 0. 0.]
Phần dư ban đầu r0: [12. -1.  5.]

Lần lặp thứ 1:
Nghiệm x = [ 2.41706161 -0.2014218   1.007109  ]
Phần dư r = [ 1.12322275  4.23696682 -1.84834123]
Chuẩn phần dư ||r|| = 4.757087609803571

Lần lặp thứ 2:
Nghiệm x = [3.07752283 0.79483181 0.71998218]
Phần dư r = [-0.2352417   0.33815995  0.63221207]
Chuẩn phần dư ||r|| = 0.7545746578244767

Lần lặp thứ 3:
Nghiệm x = [3. 1. 1.]
Phần dư r = [-1.77635684e-15  8.88178420e-16 -1.77635684e-15]
Chuẩn phần dư ||r|| = 2.6645352591003757e-15
Thỏa mãn sau 3 lần lặp.

= KIỂM CHỨNG HỘI TỤ =
Cấp ma trận n: 3
Số lần lặp k: 3
KẾT LUẬN: Thuật toán hội tụ không quá n bước.

VÍ DỤ CẤP 4
Nghiệm ban đầu x0: [0. 0. 0. 0.]
Phần dư ban đầu r0: [ 2.  4.  6. 13.]

Lần lặp thứ 1:
Nghiệm x = [0.66176471 1.32352941 1.98529412 4.30147059]
Phần dư r = [ 0.67647059  1.35294118  3.68382353 -2.22058824]
Chuẩn phần dư ||r|| = 4.559564693072292

Lần lặp thứ 2:
Nghiệm x = [0.88034986 1.76069972 3.06093067 4.04274974]
Phần dư r = 

(array([1., 2., 3., 4., 5.]), 5)

### Điều kiện dừng theo sai số

In [74]:
import numpy as np

def conjugate_gradient(A, b, x0, eps) -> tuple[np.ndarray, int]:
    """
    A: Ma trận vuông, đối xứng xác định dương.
    b: Vectơ vế phải.
    x0: Vectơ nghiệm ban đầu.
    eps: Sai số cho phép, eps > 0.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.asarray(x0, dtype=float).copy()
    chuan_b = np.sqrt(np.sum(b**2))

    # Kiểm tra ma trận vuông
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A phải là ma trận vuông không rỗng.")

    n = A.shape[0]

    # Kiểm tra ma trận đối xứng xác định dương
    if not np.allclose(A, A.T):
        raise ValueError("A phải là ma trận đối xứng.")
    for i in range(1, n + 1):
        det = np.linalg.det(A[:i, :i])

        if det <= 0:
            raise ValueError("A không phải ma trận xác định dương.")    

    # Khởi tạo phần dư và hướng tìm kiếm
    r = b - np.dot(A, x)
    s = r.copy()

    print("Nghiệm ban đầu x0:", x)
    print("Phần dư ban đầu r0:", r)

    if np.max(np.abs(r)) < eps:
        print("Nghiệm ban đầu đã thỏa mãn điều kiện dừng.")
        return x, 0

    for k in range(1, n + 1):

        # Tính alpha
        As = np.dot(A, s)
        mau = np.dot(s, As)

        if mau <= 0 or not np.isfinite(mau):
            raise ValueError("Mẫu số không hợp lệ.")

        alpha = np.dot(s, r) / mau

        # Cập nhật nghiệm và phần dư
        x = x + alpha * s
        r_moi = b - np.dot(A, x)
        chuan_r = np.sqrt(np.sum(r_moi**2))
        err = chuan_r / chuan_b

        # Hiển thị kết quả từng lần lặp
        print(f"\nLần lặp thứ {k}:")
        print("Nghiệm x =", x)
        print("Phần dư r =", r_moi)
        print("Sai số", err)

        # Kiểm tra điều kiện dừng
        if err < eps:
            print(f"Thỏa mãn sau {k} lần lặp.")
            return x, k

        # Tính beta
        beta = -np.dot(r_moi, np.dot(A, s)) / mau

        # Cập nhật hướng tìm kiếm
        s = r_moi + beta * s
        r = r_moi.copy()

    print(f"Không hội tụ sau {n} lần lặp.")
    return x, n


In [75]:
A = np.array([[4, -1,  1],
              [-1, 4, -2],
              [1, -2,  4]], dtype=float)
b = np.array([12, -1, 5], dtype=float)
kq = conjugate_gradient(A, b, x0 = np.array([0, 0, 0]) , eps = 1e-8)

Nghiệm ban đầu x0: [0. 0. 0.]
Phần dư ban đầu r0: [12. -1.  5.]

Lần lặp thứ 1:
Nghiệm x = [ 2.41706161 -0.2014218   1.007109  ]
Phần dư r = [ 1.12322275  4.23696682 -1.84834123]
Sai số 0.36485196455989594

Lần lặp thứ 2:
Nghiệm x = [3.07752283 0.79483181 0.71998218]
Phần dư r = [-0.2352417   0.33815995  0.63221207]
Sai số 0.05787323440228581

Lần lặp thứ 3:
Nghiệm x = [3. 1. 1.]
Phần dư r = [-1.77635684e-15  8.88178420e-16 -1.77635684e-15]
Sai số 2.0436052552793447e-16
Thỏa mãn sau 3 lần lặp.
